# 03 · Disease risk (healthy vs diseased)

**What changed from the original Part 3 notebook, and why it matters**

| Original | Problem | Now |
|---|---|---|
| Features from a shuffled generator, labels from `.labels` | Features and labels were in different orders, so every model learned from scrambled labels (validation AUC around 0.5 = coin flip) | `tf.data` shuffles image and label **together** |
| Test set used as `validation_data` with early stopping | Test score was used to pick the model, so it was optimistic | Official validation set for early stopping and thresholds; test set used **once** at the end |
| Features extracted once from augmented images | Each image got one frozen random augmentation | Fresh random augmentation every epoch, on training images only |
| `rescale=1/255` for every backbone | EfficientNet expects 0-255, ResNeXt preprocessing was never applied; both collapsed to predicting "diseased" for every image | Each model does its own correct scaling internally |
| Accuracy as the headline metric | 79% of images are diseased, so 79% accuracy is the do-nothing baseline | AUC, sensitivity, specificity, PPV/NPV with bootstrap 95% CIs, plus the baseline |
| Imbalance noted but not handled | Model leaned toward "diseased" | Class-weighted loss and a threshold set for 90% sensitivity on validation |
| No ensemble despite the title | — | Three backbones averaged, weighted by validation AUC |
| No explanation of predictions | — | Grad-CAM heatmaps |

Runtime on a Colab T4: roughly 15-25 minutes per backbone.

In [ ]:
# --- Setup (run once per session) -------------------------------------------
# On Colab: mounts Google Drive, clones this repo and installs requirements.
# Locally: just makes the `iluminado` package importable.
import os, sys, subprocess

REPO_URL = "https://github.com/cathykam2006/Capstone-Project-Illuminado-Retinal--Image-Classification.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    else:   # always use the latest code from GitHub
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

REPO_DIR = next(p for p in [os.getcwd(), os.path.dirname(os.getcwd())] if os.path.isdir(os.path.join(p, "iluminado")))
sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)

In [ ]:
# --- Config: edit these paths ---------------------------------------------------
# PROJECT_DIR holds the cached images, trained models and results. On Colab it
# lives on your Google Drive so it survives between sessions.
PROJECT_DIR = "/content/drive/MyDrive/iluminado" if IN_COLAB else os.path.join(REPO_DIR, "work")
LABEL_DIR = os.path.join(REPO_DIR, "data")            # the three RFMiD label CSVs (in the repo)
MANIFEST = os.path.join(PROJECT_DIR, "manifest.csv")  # written by notebook 01
os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
# --- Training config ---------------------------------------------------------------
IMG_SIZE = 384        # fundus lesions are small; 384 keeps far more detail than the old 150-256
BATCH_SIZE = 16
BACKBONES = ["efficientnet_b0", "densenet121", "convnext_tiny"]   # different families -> more useful ensemble
HEAD_EPOCHS, FINETUNE_EPOCHS = 5, 30
SEED = 42

import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "none (training will be very slow; in Colab use Runtime > Change runtime type > GPU)")
if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")   # ~2x faster on T4/A100

from iluminado.data import make_dataset, TARGET_DISEASES
from iluminado import models as M, evaluate as E
m = pd.read_csv(MANIFEST)
if IN_COLAB:   # train from local disk, not Google Drive: many times faster
    from iluminado.data import localize
    m = localize(m)
MODEL_DIR = os.path.join(PROJECT_DIR, "models"); os.makedirs(MODEL_DIR, exist_ok=True)
RESULTS_DIR = os.path.join(PROJECT_DIR, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)

## Data

In [ ]:
TASK = "binary"
LABELS = ["Disease_Risk"]
split = {s: m[m.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y = {s: split[s][LABELS].to_numpy("float32") for s in split}

train_ds = make_dataset(split["train"].path, y["train"], IMG_SIZE, BATCH_SIZE, training=True, seed=SEED)
# Evaluation datasets are NOT shuffled, so predictions line up with y["val"] / y["test"].
val_ds = make_dataset(split["val"].path, y["val"], IMG_SIZE, BATCH_SIZE, training=False)
test_ds = make_dataset(split["test"].path, y["test"], IMG_SIZE, BATCH_SIZE, training=False)

pos_weight = M.positive_weights(y["train"])
print({s: f"{len(v)} images, {v.mean():.1%} diseased" for s, v in y.items()})
print("positive-class weight:", pos_weight.round(3))

## Train each backbone

Models are saved to Drive as they finish, so if Colab disconnects you can re-run this cell and finished backbones are skipped. Validation and test predictions are averaged over each image and its horizontal and vertical flips (test-time augmentation).

In [ ]:
probs = {"val": {}, "test": {}}
for bb in BACKBONES:
    path = os.path.join(MODEL_DIR, f"{TASK}_{bb}.keras")
    if os.path.exists(path):
        print(f"[{bb}] found saved model, skipping training")
        model = M.load(path)
    else:
        print(f"\n===== {bb} =====")
        keras.backend.clear_session()
        model = M.build_model(bb, IMG_SIZE, n_outputs=1)
        h1, h2 = M.train(model, train_ds, val_ds, pos_weight, n_outputs=1, ckpt_path=path,
                         head_epochs=HEAD_EPOCHS, finetune_epochs=FINETUNE_EPOCHS)
        model = M.load(path)   # best validation-AUC checkpoint
        pd.DataFrame(h2.history).to_csv(os.path.join(RESULTS_DIR, f"{TASK}_{bb}_history.csv"), index=False)
    for s, ds in (("val", val_ds), ("test", test_ds)):
        probs[s][bb] = E.predict(model, ds)[:, 0]
    np.save(os.path.join(RESULTS_DIR, f"{TASK}_{bb}_val.npy"), probs["val"][bb])
    np.save(os.path.join(RESULTS_DIR, f"{TASK}_{bb}_test.npy"), probs["test"][bb])

## Classical baseline (no deep learning)

34 simple image statistics (colour, edges, bright/dark pixel shares, brightness by ring, sharpness) fed to logistic regression, a random forest and gradient boosting, probabilities averaged. Trained on the training set only, scored like everything else. The CNNs need to clearly beat this to justify themselves.

In [ ]:
from sklearn.metrics import roc_auc_score
from iluminado.baseline import features_for, fit_predict
feats = {s: features_for(split[s].path.tolist()) for s in split}
pv, pt = fit_predict(feats["train"], y["train"], [feats["val"], feats["test"]])
probs["val"]["classical baseline"], probs["test"]["classical baseline"] = pv[:, 0], pt[:, 0]
print("classical baseline validation AUC:", round(roc_auc_score(y["val"][:, 0], pv[:, 0]), 3))

## Ensemble and thresholds (validation set only)

- Ensemble weights are each model's validation AUC.
- The decision threshold is chosen on validation so that **90% of diseased eyes are flagged** (screening setting: missing disease is worse than a false alarm). It is then applied unchanged to the test set.

In [ ]:
from sklearn.metrics import roc_auc_score
yv, yt = y["val"][:, 0], y["test"][:, 0]
val_auc = {bb: roc_auc_score(yv, p) for bb, p in probs["val"].items()}
probs["val"]["ensemble"] = E.ensemble({k: probs["val"][k] for k in BACKBONES}, val_auc)
probs["test"]["ensemble"] = E.ensemble({k: probs["test"][k] for k in BACKBONES}, val_auc)

TARGET_SENSITIVITY = 0.90
thresholds = {k: E.threshold_for_sensitivity(yv, p, TARGET_SENSITIVITY) for k, p in probs["val"].items()}
pd.Series(val_auc, name="validation AUC").round(3)

## Final test results (test set used once)

In [ ]:
rows = {}
for k, p in probs["test"].items():
    r = E.binary_report(yt, p, thresholds[k])
    lo, hi = E.bootstrap_ci(yt, p)
    r["AUC 95% CI"] = f"{lo:.3f}-{hi:.3f}"
    rows[k] = r
results = pd.DataFrame(rows).T
cols = ["AUC", "AUC 95% CI", "PR-AUC", "sensitivity", "specificity", "PPV", "NPV", "balanced_acc", "accuracy", "majority_baseline_acc"]
results[cols].to_csv(os.path.join(RESULTS_DIR, f"{TASK}_test_results.csv"))
results[cols]

In [ ]:
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for k, p in probs["test"].items():
    fpr, tpr, _ = roc_curve(yt, p)
    axes[0].plot(fpr, tpr, lw=2.5 if k == "ensemble" else 1.2, label=f"{k} (AUC {roc_auc_score(yt, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
axes[0].set(xlabel="1 - specificity", ylabel="sensitivity", title="ROC on held-out test set"); axes[0].legend(fontsize=8)
pred = (probs["test"]["ensemble"] >= thresholds["ensemble"]).astype(int)
ConfusionMatrixDisplay.from_predictions(yt, pred, display_labels=["Healthy", "Diseased"], ax=axes[1], colorbar=False)
axes[1].set_title(f"Ensemble at threshold {thresholds['ensemble']:.2f}")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"{TASK}_roc_cm.png"), dpi=150); plt.show()

## Does the ensemble beat its best single model?

Both are scored on the same resampled test images. If the 95% interval of the difference excludes 0, the gain is real; if not, a single model is simpler and just as good, and that is worth saying in the README.

In [ ]:
best_single = max(BACKBONES, key=lambda k: val_auc[k])      # chosen on validation, not test
rows = []
for other in [best_single, "classical baseline"]:
    d, lo, hi, verdict = E.paired_bootstrap_diff(yt, probs["test"]["ensemble"], probs["test"][other])
    rows.append({"ensemble minus": other, "AUC difference": round(d, 3), "95% CI": f"{lo:+.3f} to {hi:+.3f}",
                 "verdict": verdict.replace("A ", "ensemble ")})
pd.DataFrame(rows)

## What is the model looking at? (Grad-CAM)

Red areas drove the "diseased" score. Check that they fall on the retina (lesions, optic disc, vessels) rather than on the black border or camera artefacts.

In [ ]:
from PIL import Image
from iluminado.gradcam import gradcam, overlay
best = max(BACKBONES, key=lambda k: val_auc[k])
model = M.load(os.path.join(MODEL_DIR, f"{TASK}_{best}.keras"))
sample = pd.concat([split["test"][yt == 1].sample(3, random_state=1), split["test"][yt == 0].sample(2, random_state=1)])
fig, axes = plt.subplots(2, len(sample), figsize=(3.2 * len(sample), 6.5))
for i, (_, row) in enumerate(sample.iterrows()):
    img = np.asarray(Image.open(row.path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype="float32")
    p = float(model.predict(img[None], verbose=0)[0, 0])
    axes[0, i].imshow(img.astype("uint8")); axes[0, i].set_title(f"#{row.ID} truth={'D' if row.Disease_Risk else 'H'}", fontsize=9)
    axes[1, i].imshow(overlay(img, gradcam(model, img))); axes[1, i].set_title(f"{best}: p={p:.2f}", fontsize=9)
    for a in axes[:, i]: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"{TASK}_gradcam.png"), dpi=150); plt.show()

## Save the ensemble config for the app

In [ ]:
config = {
    "task": TASK, "img_size": IMG_SIZE, "labels": LABELS,
    "models": {bb: f"{TASK}_{bb}.keras" for bb in BACKBONES},
    "weights": {bb: float(val_auc[bb]) for bb in BACKBONES},
    "threshold": float(thresholds["ensemble"]),
    "threshold_rule": f"{TARGET_SENSITIVITY:.0%} sensitivity on validation set",
    "test_auc": float(results.loc["ensemble", "AUC"]),
}
with open(os.path.join(MODEL_DIR, f"{TASK}_ensemble.json"), "w") as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))
print("\nCopy the .keras files and this JSON into the repo's models/ folder for the Streamlit app.")

In [ ]:
# Markdown table for the README
tbl = results[cols].copy()
num = [c for c in cols if c != "AUC 95% CI"]
tbl[num] = tbl[num].astype(float).round(3)
print(tbl.to_markdown())